# Biohub - 3D Cell Detection Local CV & Dual 3D Interactive Dashboard

このノートブックは、**Biohub - Cell Tracking During Development** コンペにおいて、**細胞検出(Cell Detection)の精度向上**に特化したローカルCV評価環境および完全統一操作性の3Dデュアルダッシュボードです。

## 主な機能
1. **オフライン Zarr インストール**: ユーザー指定のオフラインwheelによる安全なセットアップ
2. **検出単体CV評価 (Detection CV Metric)**: ハンガリアン法による正解(GT)と予測の1対1マッチングで Precision, Recall, F1-score を自動計算
3. **操作性＆座標軸完全統一 3Dデュアルダッシュボード (Dual 3D Subplot Dashboard)**: 
   - **左**: 3D立体回転点群グラフ (GT:緑の球 vs Pred:赤の×点)
   - **右**: 3D MIPサーフェス重ね合わせグラフ (左と全く同じ操作性 ＆ 左下原点・軸向き完全統一)

In [ ]:
# Install zarr from the user's offline dataset
# ユーザーが準備したオフラインデータセットから zarr をインストール
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr

In [ ]:
%matplotlib inline
import os
import glob
import zarr
import numpy as np
import pandas as pd
from skimage.feature import blob_dog
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
from tqdm import tqdm

# Enable Plotly renderer for Kaggle Notebook
init_notebook_mode(connected=True)

## 1. Setup & Data Path Resolution
## 1. 設定とデータパスの確認

In [ ]:
# Candidate data paths in Kaggle environment
CANDIDATES = [
    "/kaggle/input/biohub-cell-tracking-during-development",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
]

ROOT_DIR = "/kaggle/input/biohub-cell-tracking-during-development"
for p in CANDIDATES:
    if os.path.exists(os.path.join(p, "train")) or os.path.exists(os.path.join(p, "test")):
        ROOT_DIR = p
        break

TRAIN_DIR = os.path.join(ROOT_DIR, "train")
TEST_DIR = os.path.join(ROOT_DIR, "test")
print(f"Using ROOT_DIR: {ROOT_DIR}")
print(f"Train directory exists: {os.path.exists(TRAIN_DIR)}")

## 2. 3D Cell Detector & Local CV Evaluation Metric Functions
## 2. 3D細胞検出モデル & 検出CV評価メトリクス関数

In [ ]:
def detect_cells_3d(image_3d, min_sigma=2, max_sigma=5, threshold=0.05):
    """Detect 3D cell centroid coordinates (z, y, x) from a 3D volume image.
    3Dボリューム画像から細胞の重心座標(z, y, x)を検出します。
    """
    img_min = image_3d.min()
    img_max = image_3d.max()
    if img_max > img_min:
        img_norm = (image_3d.astype(np.float32) - img_min) / (img_max - img_min)
    else:
        img_norm = np.zeros_like(image_3d, dtype=np.float32)
        
    blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
    if len(blobs) > 0:
        return blobs[:, :3]  # (z, y, x)
    return np.empty((0, 3))

def evaluate_detection_cv(pred_coords, gt_coords, radius_threshold=5.0):
    """Evaluate detection performance using 1-to-1 matching (Hungarian Algorithm).
    正解(GT)座標と予測座標の1対1マッチングを行い、Precision, Recall, F1-score を算出します。
    """
    if len(pred_coords) == 0 and len(gt_coords) == 0:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "tp": 0, "fp": 0, "fn": 0}
    if len(pred_coords) == 0 or len(gt_coords) == 0:
        fp = len(pred_coords)
        fn = len(gt_coords)
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "fp": fp, "fn": fn}
    
    # Compute pairwise Euclidean distances
    dists = cdist(pred_coords, gt_coords)
    row_ind, col_ind = linear_sum_assignment(dists)
    
    # Count matches within radius_threshold
    matched_dists = dists[row_ind, col_ind]
    tp = np.sum(matched_dists <= radius_threshold)
    
    fp = len(pred_coords) - tp
    fn = len(gt_coords) - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {"precision": precision, "recall": recall, "f1": f1, "tp": int(tp), "fp": int(fp), "fn": int(fn)}

## 3. Dual 3D Interactive Subplot Dashboard (Unified Origin & Coordinate Axes)
## 3. 操作性＆座標原点・軸向き完全統一 3Dデュアルダッシュボード

In [ ]:
def plot_detection_dashboard(image_3d, gt_coords, pred_coords, dataset_name="", metrics=None):
    """Dual 3D Subplot Dashboard with identical 3D mouse controls AND unified origin (Bottom-Left).
    左右両方の図で「ドラッグ回転・ホイール拡大縮小・左下原点(0,0)」を完全統一した3Dダッシュボード。
    """
    mip = np.max(image_3d, axis=0)
    ny, nx = mip.shape
    x_grid, y_grid = np.meshgrid(np.arange(nx), np.arange(ny))
    z_zero = np.zeros_like(mip)
    
    # Both subplots are type: 'scene' (3D Scene)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(
            "1. 3D Point Cloud (Bottom-Left Origin)",
            "2. 3D MIP Surface Overlay (Bottom-Left Origin)"
        )
    )
    
    # ==========================================
    # --- Left: 3D Point Cloud (GT vs Pred) ---
    # ==========================================
    if len(gt_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=gt_coords[:, 2], y=gt_coords[:, 1], z=gt_coords[:, 0],
            mode='markers',
            name='GT (3D)',
            marker=dict(size=4, color='lime', opacity=0.8, symbol='circle'),
            text=[f"GT (Z={z:.1f}, Y={y:.1f}, X={x:.1f})" for z, y, x in gt_coords],
            hoverinfo='text'
        ), row=1, col=1)
        
    if len(pred_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=pred_coords[:, 2], y=pred_coords[:, 1], z=pred_coords[:, 0],
            mode='markers',
            name='Pred (3D)',
            marker=dict(size=3, color='red', opacity=0.9, symbol='x'),
            text=[f"Pred (Z={z:.1f}, Y={y:.1f}, X={x:.1f})" for z, y, x in pred_coords],
            hoverinfo='text'
        ), row=1, col=1)
        
    # ==========================================
    # --- Right: 3D MIP Surface Overlay (Unified Origin) ---
    # ==========================================
    # 1. Base 3D Surface for MIP image at Z=0
    fig.add_trace(go.Surface(
        x=x_grid, y=y_grid, z=z_zero,
        surfacecolor=mip,
        colorscale='Gray',
        showscale=False,
        hoverinfo='x+y+z',
        name='MIP Surface'
    ), row=1, col=2)
    
    # 2. GT nodes projected on 3D Surface
    if len(gt_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=gt_coords[:, 2], y=gt_coords[:, 1], z=np.zeros(len(gt_coords)) + 0.5,
            mode='markers',
            name='GT (MIP)',
            marker=dict(size=5, color='lime', opacity=0.9, symbol='circle'),
            text=[f"GT Cell (Z={z:.1f}, Y={y:.1f}, X={x:.1f})" for z, y, x in gt_coords],
            hoverinfo='text'
        ), row=1, col=2)
        
    # 3. Pred nodes projected on 3D Surface
    if len(pred_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=pred_coords[:, 2], y=pred_coords[:, 1], z=np.zeros(len(pred_coords)) + 0.5,
            mode='markers',
            name='Pred (MIP)',
            marker=dict(size=4, color='red', opacity=0.9, symbol='x'),
            text=[f"Pred Cell (Z={z:.1f}, Y={y:.1f}, X={x:.1f})" for z, y, x in pred_coords],
            hoverinfo='text'
        ), row=1, col=2)
        
    f1_str = f" - Local CV F1: {metrics['f1']:.4f}" if metrics else ""
    fig.update_layout(
        title_text=f"Cell Detection Dual 3D Dashboard [{dataset_name}]{f1_str}",
        scene1=dict(
            xaxis=dict(title='X (Width)'),
            yaxis=dict(title='Y (Height)'),
            zaxis=dict(title='Z (Depth)'),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        scene2=dict(
            xaxis=dict(title='X (Width)'),
            yaxis=dict(title='Y (Height)'),  # 左下原点に統一 (autorange='reversed'を削除)
            zaxis=dict(title='Z=0 (MIP Surface)'),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.2),
            camera=dict(eye=dict(x=0, y=0, z=2.2))  # 真上視点(原点統一)
        ),
        height=650,
        margin=dict(l=20, r=20, b=20, t=60)
    )
    
    fig.show(renderer="iframe_connected")

## 4. Main Execution: Run Detection CV & Render Visualizations
## 4. メイン処理: 検出CV評価の実行と可視化プロットの表示

In [ ]:
# Find available Zarr datasets in train or test folder
zarr_paths = glob.glob(os.path.join(TRAIN_DIR, "*.zarr"))
if len(zarr_paths) == 0:
    zarr_paths = glob.glob(os.path.join(TEST_DIR, "*.zarr"))

if len(zarr_paths) == 0:
    print(f"Warning: No Zarr datasets found under {ROOT_DIR}. Generating synthetic 3D volume for validation.")
    img_3d = np.zeros((30, 128, 128), dtype=np.float32)
    gt_coords = np.array([[10, 40, 50], [20, 80, 90], [15, 60, 30]], dtype=float)
    for (z, y, x) in gt_coords:
        img_3d[int(z), int(y), int(x)] = 1.0
    dataset_name = "Synthetic Sample"
else:
    target_zarr = zarr_paths[0]
    dataset_name = os.path.basename(target_zarr)
    print(f"Loading dataset: {dataset_name}")
    store = zarr.open(target_zarr, mode='r')
    arr = store['0']
    
    has_channels = (arr.ndim == 5)
    if has_channels:
        img_3d = arr[0, 0, :, :, :]
    else:
        img_3d = arr[0, :, :, :]
        
    gt_csv = target_zarr.replace(".zarr", "_nodes.csv")
    if os.path.exists(gt_csv):
        df_gt = pd.read_csv(gt_csv)
        df_gt_t0 = df_gt[df_gt['t'] == 0]
        gt_coords = df_gt_t0[['z', 'y', 'x']].values
    else:
        print("GT CSV not found, using high-confidence blob detection as pseudo Ground Truth for demo.")
        gt_coords = detect_cells_3d(img_3d, min_sigma=2, max_sigma=5, threshold=0.08)

print(f"\n--- [Dataset: {dataset_name}] ---")
print(f"3D Image Shape: {img_3d.shape}")
print(f"Ground Truth (GT) Cell Count: {len(gt_coords)}")

# 1. Run 3D Cell Detection
pred_coords = detect_cells_3d(img_3d, min_sigma=2, max_sigma=5, threshold=0.05)
print(f"Predicted Cell Count: {len(pred_coords)}")

# 2. Calculate Detection Local CV Metrics
metrics = evaluate_detection_cv(pred_coords, gt_coords, radius_threshold=5.0)
print("\n=================== LOCAL CV RESULTS ===================")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print(f"  F1-Score:  {metrics['f1']:.4f}")
print(f"  (TP: {metrics['tp']}, FP: {metrics['fp']}, FN: {metrics['fn']})")
print("========================================================\n")

# 3. Render Dual 3D Interactive Subplot Dashboard (Unified Origin & Axis Directions)
plot_detection_dashboard(img_3d, gt_coords, pred_coords, dataset_name=dataset_name, metrics=metrics)